# IMSRG Brillouin Generator (η)

This notebook uses `qcombo.easyCombo` to reproduce the expression of the **Brillouin generator** in the IMSRG method.

## Theoretical Background

In IMSRG (In-Medium Similarity Renormalization Group), the Brillouin generator is defined as:

$$\eta = [H, A] $$

where $H$ is the Hamiltonian (including one-body term $f$ and two-body term $\Gamma$), and $A$ is the generator. We need to compute:

- **One-body term** ($\eta_l^k$): $[H, \tilde{A}_l^k]$  →  `easyCombo(1, 1)` 
- **Two-body term** ($\eta_{mn}^{kl}$): $[H, \tilde{A}_{mn}^{kl}]$  →  `easyCombo(2, 2)` + `easyCombo(1, 2)` nested

**Reference**: H. Hergert, **In-Medium Similarity Renormalization Group for Closed and Open-Shell Nuclei**, Eq. (102)-(103)

In [1]:
# Import qcombo and necessary utility functions
import qcombo
from IPython.display import display, Latex
from sympy import IndexedBase, symbols
from sympy import preorder_traversal
from sympy.tensor.indexed import Indexed
from sympy.core.mul import Mul
from sympy.core.add import Add

import time

# Define tensor symbols
A = IndexedBase('A')       # generator
G = IndexedBase('G')       # left operator (in easyCombo)
H = IndexedBase('H')       # right operator (in easyCombo)
R = IndexedBase('R')       # nested commutator result
f = IndexedBase('f')       # one-body matrix element
Gamma = IndexedBase(915)   # Γ two-body matrix element
lamda = IndexedBase(955)   # λ (lambda) density matrix
delta = IndexedBase(948)   # δ Kronecker delta
n = IndexedBase('n')       # occupation number

k= symbols('k')
l= symbols('l')
m= symbols('m')
n= symbols('n')


print(f"qcombo version: {qcombo.__version__}")

qcombo version: 0.2.0


In [2]:
# Helper function: display SymPy expressions as LaTeX in Jupyter
def jupyterDisplay(expr, title=None):
    """
    Display SymPy expression in LaTeX format in Jupyter Notebook
    """
    if expr == 0 or expr is None:
        display(Latex(f"$$0$$"))
        return
    latex_expr = qcombo.texExp(expr)
    if title:
        print(title)
    display(Latex(f"$${latex_expr}$$"))

---
## 1. One-body Term: $\eta_l^k \equiv \langle\Phi|[H, \tilde{A}_l^k]|\Phi\rangle$


### Computation strategy:
- Use easyCombo to compute $[\sum_{ab} G^{a}_{b}, \sum_{ij} H^{i}_{j}]$ and $[\sum_{abcd} G^{ab}_{cd}, \sum_{ij} H^{i}_{j}]$
- Corresponding respectively to $\sum_{kl} T^k_l \eta_l^k \equiv \langle\Phi|[f+\Gamma, \sum_{lk} T^l_k\tilde{A}_l^k]_{0B}|\Phi\rangle = [f,T]_{0B} + [\Gamma,T]_{0B} $
- By using the correspondence relation, we can obtain the matrix elements of eta
- Therefore, we only need to compute easyCombo(1,1,0) and (2,1,0)


In [3]:
# Step 1: Compute [1B, 1B] commutator
print("="*60)
print("Computing [1B, 1B] commutator for one-body η term...")
print("="*60)

t0 = time.time()
comm_110 = qcombo.easyCombo(1, 1,0,parallel= False,show_process=False, savefile=False)
t1 = time.time()

print(f"\nComputation time: {t1-t0:.2f}s")
print(f"\nAvailable expression keys: {list(comm_110.expr_dict.keys())}")

Computing [1B, 1B] commutator for one-body η term...

Computation time: 0.16s

Available expression keys: ['0B_lambda1B']


In [4]:
commmutator_110_expr = 0
for key, value in comm_110.expr_dict.items():
    commmutator_110_expr += value

# jupyterDisplay(commmutator_110_expr)

In [5]:
from nt import replace


eta1B_up_idx = symbols('k')
eta1B_lo_idx = symbols('l')

def find_H_tensor(expr):
    for term in preorder_traversal(expr):
        if isinstance(term, Indexed) and term.base == IndexedBase('H'):
            return term
    return None

def find_G_tensor(expr):
    for term in preorder_traversal(expr):
        if isinstance(term, Indexed) and term.base == IndexedBase('G'):
            return term
    return None

H_tensor = find_H_tensor(commmutator_110_expr)

H_up_idx,H_lo_idx = H_tensor.indices[0],H_tensor.indices[1]
# print(H_up_idx,H_lo_idx)

replace_dict = {}

replace_dict[H_up_idx] = (eta1B_up_idx,)
replace_dict[H_lo_idx] = (eta1B_lo_idx,)

commmutator_110_expr=commmutator_110_expr.xreplace(replace_dict)

# jupyterDisplay(commmutator_110_expr)


In [6]:
def replace_G_Base(expr,newBase):
    G_tensor = find_G_tensor(expr)
    for term in preorder_traversal(expr):
        if isinstance(term, Indexed) and term.base == IndexedBase('G'):
            if type(newBase) is IndexedBase:
                return expr.xreplace({G_tensor.base: newBase})
            else:
                return expr.xreplace({G_tensor.base: IndexedBase(newBase)} )

def replace_H(expr,replce_element):
    for term in preorder_traversal(expr):
        if isinstance(term, Indexed) and term.base == IndexedBase('H'):
            return expr.xreplace({term: replce_element})



eta1B_110_expr = replace_H(replace_G_Base(commmutator_110_expr,'f'),1)
# print('the eta 1-body from 110 commutator is')
# jupyterDisplay(eta1B_110_expr)
    

In [7]:
# Step 2: Compute [2B, 1B] commutator, contract to 0-body, to obtain the Γ-related part of η_1B
# -½ Σ_abc (Γ_bc^la λ_c^ka - Γ_kc^ab λ_b^la)
print("="*60)
print("Computing [2B, 1B] -> 0B commutator for one-body η term...")
print("="*60)

t0 = time.time()
comm_210 = qcombo.easyCombo(2, 1, 0, parallel=False, show_process=False, savefile=False)
t1 = time.time()

print(f"\nComputation time: {t1-t0:.2f}s")
print(f"\nAvailable expression keys: {list(comm_210.expr_dict.keys())}")

Computing [2B, 1B] -> 0B commutator for one-body η term...

Computation time: 0.09s

Available expression keys: ['0B_lambda2B']


In [8]:
# Display raw result of [2B, 1B] -> 0B
commutator_210_expr = 0
for key, value in comm_210.expr_dict.items():
    commutator_210_expr += value

# jupyterDisplay(commutator_210_expr, "[2B, 1B] -> 0B raw result:")

In [9]:
# Replace H indices in [2B,1B] result with k,l (corresponding to η_l^k)
commutator_210_expr_replaced = commutator_210_expr.expand()

res_210 = 0
for term in commutator_210_expr_replaced.args:
    H_tensor_21 = find_H_tensor(term)
    if H_tensor_21 is not None:
        H_up_idx_21, H_lo_idx_21 = H_tensor_21.indices[0][0], H_tensor_21.indices[1][0]
        # print(H_up_idx_21)
        replace_dict_21 = {}
        replace_dict_21[H_up_idx_21] = eta1B_up_idx  # k
        replace_dict_21[H_lo_idx_21] = eta1B_lo_idx  # l
        res_210 += term.subs(replace_dict_21)

commutator_210_expr_replaced = res_210

# jupyterDisplay(commutator_210_expr_replaced, "[2B, 1B] -> 0B with H indices replaced (k,l):")

In [10]:
# Extract the Γ-related part of η_1B: G -> Γ, H -> 1
# the coeficent comes from Gamma
eta1B_210_expr = replace_H(replace_G_Base(commutator_210_expr_replaced, r'\Gamma'), 1)/4
# print('the eta 1-body from 210 commutator (Gamma contribution) is:')
# jupyterDisplay(eta1B_210_expr)

In [11]:
# Combine the complete η_1B = [f contribution] + [Γ contribution]
print("="*60)
print("Complete η_1B (one-body Brillouin generator):")
print("="*60)

eta1B_full = eta1B_110_expr + eta1B_210_expr
jupyterDisplay(eta1B_full, "η_l^k = [f, A]_0B + [Γ, A]_0B:")

Complete η_1B (one-body Brillouin generator):
η_l^k = [f, A]_0B + [Γ, A]_0B:


<IPython.core.display.Latex object>

### Verify that η_1B matches the target formula

Target formula: $$\eta_l^k = f_l^k(n_l - n_k) - \frac{1}{2}\sum_{abc}\left(\Gamma_{bc}^{la}\lambda_{bc}^{ka} - \Gamma_{kc}^{ab}\lambda_{lc}^{ab}\right)$$

---
## 2. Two-body Term: $\eta_{mn}^{kl} \equiv \langle\Phi|[H, \tilde{A}_{mn}^{kl}]|\Phi\rangle$


### Computation strategy:
- **2B η term decomposition**: $\eta_{mn}^{kl}$ comes from the following commutators:
  - `easyCombo(2, 2, 0)`: $[\Gamma, \tilde{A}]$ contracted to 2-body
  - `easyCombo(1, 2, 0)`: $[f, \tilde{A}]$ contracted to 2-body (via nested commutator)
- Need to compute multiple λ body classifications (λ_1B, λ_2B, λ_3B)


In [12]:
# Step 3: Compute [2B, 2B] commutator, contract to 0-body
# This is the Γ-related part of η_2B (Γ terms)
print("="*60)
print("Computing [2B, 2B] -> 0B commutator for two-body η term...")
print("="*60)

t0 = time.time()
comm_220 = qcombo.easyCombo(2, 2, 0, parallel=False, show_process=False, savefile=False)
t1 = time.time()

print(f"\nComputation time: {t1-t0:.2f}s")
# print(f"\nAvailable expression keys: {list(comm_220.expr_dict.keys())}")

Computing [2B, 2B] -> 0B commutator for two-body η term...

Computation time: 1.37s


In [13]:
# Display [2B, 2B] -> 0B results classified by λ body number
full_comm_220_expr = 0
for key, value in comm_220.expr_dict.items():
    # print(key)
    # jupyterDisplay(value)
    full_comm_220_expr+=value

# print('full commutator 220 expression :')
# jupyterDisplay(full_comm_220_expr)



In [14]:
# Define indices for η_2B: η_{mn}^{kl}
eta2B_up_indices = (symbols('k'),symbols('l'))
eta2B_lo_indices = (symbols('m'),symbols('n'))

def replace_H2B_indices_mul(expr,new_up_idx,new_lo_idx):
    if isinstance(expr, Mul):
        replace_dict = {}
        for term in preorder_traversal(expr):
            if isinstance(term, Indexed) and term.base == IndexedBase('H'):
                H_tensor = term
                H_up_left_idx = H_tensor.indices[0][0]
                H_up_right_idx = H_tensor.indices[0][1]
                H_lo_left_idx = H_tensor.indices[1][0]
                H_lo_right_idx = H_tensor.indices[1][1]
                # replace
                replace_dict[H_up_left_idx] = new_up_idx[0]
                replace_dict[H_up_right_idx] = new_up_idx[1]
                replace_dict[H_lo_left_idx] = new_lo_idx[0]
                replace_dict[H_lo_right_idx] = new_lo_idx[1]
                return expr.xreplace(replace_dict)
    elif isinstance(expr, Add):
        res = 0
        for arg in expr.args:
            res += replace_H2B_indices_mul(arg, new_up_idx, new_lo_idx)
        return res
    else:
        return expr




full_comm_220_expr = replace_H2B_indices_mul(full_comm_220_expr.expand(),eta2B_up_indices,eta2B_lo_indices)
full_comm_220_expr = qcombo.simplify.MergeSameMatrixElement(full_comm_220_expr)
# jupyterDisplay(full_comm_220_expr)


In [15]:
# Extract the Γ contribution part of η_2B from [2B, 2B] -> 0B
# G -> Γ, H -> 1
def find_H_tensor(expr):
    for term in preorder_traversal(expr):
        if isinstance(term, Indexed) and term.base == IndexedBase('H'):
            return term
    return None

def find_G_tensor(expr):
    for term in preorder_traversal(expr):
        if isinstance(term, Indexed) and term.base == IndexedBase('G'):
            return term
    return None

def replace_H(expr,replce_element):
    for term in preorder_traversal(expr):
        if isinstance(term, Indexed) and term.base == IndexedBase('H'):
            return expr.xreplace({term: replce_element})

def replace_G_Base(expr,newBase):
    G_tensor = find_G_tensor(expr)
    for term in preorder_traversal(expr):
        if isinstance(term, Indexed) and term.base == IndexedBase('G'):
            if type(newBase) is IndexedBase:
                return expr.xreplace({G_tensor.base: newBase})
            else:
                return expr.xreplace({G_tensor.base: IndexedBase(newBase)} )
    
# the factor 1/4 comes from Gamma
eta2B_220_expr = replace_H(replace_G_Base(full_comm_220_expr, r'\Gamma'), 1)/4
# jupyterDisplay(eta2B_220_expr)


In [16]:
# Step 4: Compute [1B, 2B] commutator, contract to 0-body
# This is the f-related part of η_2B (f terms)
print("="*60)
print("Computing [1B, 2B] -> 0B commutator for two-body η term...")
print("="*60)

t0 = time.time()
comm_120 = qcombo.easyCombo(1, 2, 0, parallel=False, show_process=False, savefile=False)
t1 = time.time()

print(f"\nComputation time: {t1-t0:.2f}s")
# print(f"\nAvailable expression keys: {list(comm_120.expr_dict.keys())}")

Computing [1B, 2B] -> 0B commutator for two-body η term...

Computation time: 0.14s


In [17]:
# Display [1B, 2B] -> 2B results classified by λ body number
full_comm_120_expr = 0
for key, value in comm_120.expr_dict.items():
    # print(key)
    # jupyterDisplay(value)
    full_comm_120_expr+=value

# print('full commutator 120 expression :')
# jupyterDisplay(full_comm_120_expr)

In [18]:
# Extract the f contribution part of η_2B from [1B, 2B] -> 0B
# G -> f, H -> 1
full_comm_120_expr = replace_H2B_indices_mul(full_comm_120_expr.expand(),eta2B_up_indices,eta2B_lo_indices)
full_comm_120_expr = qcombo.simplify.MergeSameMatrixElement(full_comm_120_expr)
# jupyterDisplay(full_comm_120_expr)


In [19]:
eta2B_120_expr = replace_H(replace_G_Base(full_comm_120_expr, 'f'), 1)
# jupyterDisplay(eta2B_120_expr)

In [20]:
# Combine the complete η_2B
print("="*60)
print("Complete η_2B (two-body Brillouin generator):")
print("="*60)

eta2B_expr = eta2B_120_expr + eta2B_220_expr
for lambdaBody in [1,2,3]:
    print(f"eta2B expresiion with {lambdaBody} part:")
    jupyterDisplay(qcombo.simplify.filterLambdaBody(eta2B_expr,lambdaBody,show_process=False))


Complete η_2B (two-body Brillouin generator):
eta2B expresiion with 1 part:


<IPython.core.display.Latex object>

eta2B expresiion with 2 part:


<IPython.core.display.Latex object>

eta2B expresiion with 3 part:


<IPython.core.display.Latex object>

### Verify that η_2B matches the target formula

Note: qcombo's output may use different index naming conventions; manual inspection of term correspondence is required.

Target formula (Eq. 1.102):
$$\begin{aligned}
\eta_{mn}^{kl} &= \Gamma_{kl}^{mn}(\bar{n}_k\bar{n}_ln_ln_n - n_kn_l\bar{n}_m\bar{n}_n) \\
&\quad + \sum_a\left(f_k^a\lambda_{mn}^{al} + f_l^a\lambda_{mn}^{ka} - f_a^m\lambda_{an}^{kl} - f_a^n\lambda_{ma}^{kl}\right) \\
&\quad +\frac{1}{2}\sum_{abc}\left[(1-\hat{P}_{mn})\Gamma_{bc}^{ma}\lambda_{bcn}^{akl} + (1-\hat{P}_{kl})\Gamma_{lc}^{ab}\lambda_{cmn}^{abk}\right] \\
&\quad +\frac{1}{2}\left[(\lambda\Gamma)_{kl}^{mn}(1-n_k-n_l) - (\Gamma\lambda)_{kl}^{mn}(1-n_m-n_n)\right] \\
&\quad +(1-\hat{P}_{mn})(1-\hat{P}_{kl})\sum_c\Gamma_{cl}^{am}\lambda_{cn}^{ak}(n_l-n_m)
\end{aligned}$$

It is worth noting that the indices in the eta2B expression given by qcombo have not been antisymmetrized, so manual antisymmetrization is required.